# Poglavlje 7: Izrada chat aplikacija
## Brzi početak s Microsoft Foundry Models API-jem

Ovaj bilježnik prilagođen je iz [Azure OpenAI uzoraka](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) koji uključuju bilježnike za pristup [Azure OpenAI](aoai-assignment.ipynb) uslugama.

> **Napomena:** GitHub modeli bit će ukinuti krajem srpnja 2026. Ovaj bilježnik sada cilja na [Microsoft Foundry modele](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), koji nude isti katalog modela besplatan za isprobavanje i iskustvo Azure AI Inference SDK-ja.


# Pregled  
"Veliki jezični modeli su funkcije koje preslikavaju tekst u tekst. S obzirom na ulazni niz teksta, veliki jezični model pokušava predvidjeti tekst koji će slijediti"(1). Ovaj "quickstart" bilježnica će korisnicima predstaviti osnovne koncepte LLM-a, osnovne zahtjeve paketa za početak rada s AML-om, lagani uvod u dizajn upita i nekoliko kratkih primjera različitih slučajeva upotrebe. 


## Sadržaj  

[Pregled](#overview)  
[Kako koristiti OpenAI uslugu](#how-to-use-openai-service)  
[1. Kreiranje vaše OpenAI usluge](#1.-creating-your-openai-service)  
[2. Instalacija](#2.-installation)    
[3. Akreditivi](#3.-credentials)  

[Slučajevi korištenja](#use-cases)    
[1. Sažmi tekst](#1.-summarize-text)  
[2. Klasificiraj tekst](#2.-classify-text)  
[3. Generiraj nove nazive proizvoda](#3.-generate-new-product-names)  
[4. Fine Tune klasifikatora](#4.fine-tune-a-classifier)  

[Reference](#references)


### Izradite svoj prvi prompt  
Ova kratka vježba pružit će vam osnovni uvod u slanje promptova modelu u Microsoft Foundry Models za jednostavan zadatak "sažimanje".


**Koraci**:  
1. Instalirajte knjižnicu `azure-ai-inference` u svoje Python okruženje, ako to već niste učinili.  
2. Učitajte standardne pomoćne knjižnice i postavite svoje vjerodajnice za Microsoft Foundry Models.  
3. Odaberite model za svoj zadatak  
4. Kreirajte jednostavan prompt za model  
5. Pošaljite svoj zahtjev model API-ju!


### 1. Instaliraj `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Uvezi pomoćne biblioteke i stvori vjerodajnice


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Pronalaženje pravog modela  
Modeli poput GPT-4o i GPT-4o mini mogu razumjeti i generirati prirodni jezik, i dostupni su u Microsoft Foundry Models katalogu zajedno s modelima iz Meta, Mistral, Cohere i Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Dizajn upita  

"Čarolija velikih jezičnih modela je u tome što se treniranjem za minimizaciju ove pogreške predviđanja kroz ogromne količine teksta, modeli na kraju nauče koncepte korisne za ta predviđanja. Na primjer, nauče koncepte poput"(1):

* kako se pišu riječi
* kako funkcionira gramatika
* kako parafrazirati
* kako odgovarati na pitanja
* kako voditi razgovor
* kako pisati na mnogim jezicima
* kako kodirati
* itd.

#### Kako kontrolirati veliki jezični model  
"Od svih ulaznih podataka za veliki jezični model, daleko najutjecajniji je tekstualni upit(1).

Velike jezične modele može se potaknuti da proizvedu izlaz na nekoliko načina:

Uputa: Reci modelu što želiš
Dovršetak: Natjeraj model da dovrši početak onoga što želiš
Demonstracija: Pokaži modelu što želiš, bilo s:
Nekoliko primjera u upitu
Stotinama ili tisućama primjera u skupu podataka za fino podešavanje"



#### Postoje tri osnovne smjernice za kreiranje upita:

**Prikaži i reci**. Jasno naznači što želiš bilo kroz upute, primjere ili kombinaciju oboje. Ako želiš da model rangira popis stavki po abecedi ili da klasificira odlomak prema sentimentu, pokaži mu da je to ono što želiš.

**Pruži kvalitetne podatke**. Ako pokušavaš izgraditi klasifikator ili natjerati model da slijedi obrazac, pobrini se da ima dovoljno primjera. Provjeri svoje primjere — model je obično dovoljno pametan da uoči osnovne pravopisne greške i da ti odgovor, ali također može pretpostaviti da je to namjerno i to može utjecati na odgovor.

**Provjeri svoje postavke.** Postavke temperature i top_p kontroliraju koliko je model determinističan pri generiranju odgovora. Ako tražiš odgovor za koji postoji samo jedan točan odgovor, želiš postaviti ove vrijednosti niže. Ako želiš raznovrsnije odgovore, možda želiš postaviti ove vrijednosti više. Najčešća pogreška ljudi s ovim postavkama je što pretpostavljaju da su to kontrole "pametnosti" ili "kreativnosti".


Izvor: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Pošalji!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ponovite isti poziv, kako se rezultati uspoređuju?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Sažetak teksta  
#### Izazov  
Sažmite tekst dodavanjem 'tl;dr:' na kraj teksta. Primijetite kako model razumije kako obaviti niz zadataka bez dodatnih uputa. Možete eksperimentirati s detaljnijim uputama od tl;dr kako biste izmijenili ponašanje modela i prilagodili dobiveni sažetak(3).  

Nedavni radovi pokazali su značajan napredak u mnogim NLP zadacima i mjerilima kroz predtreniranje na velikom korpusu teksta, nakon čega slijedi fino podešavanje na specifičnom zadatku. Iako je arhitektura obično nezavisna od zadatka, ova metoda još uvijek zahtijeva skupove podataka za fino podešavanje s tisućama ili desetinama tisuća primjera. Nasuprot tome, ljudi uglavnom mogu obaviti novi jezični zadatak s nekoliko primjera ili jednostavnih uputa - što trenutni NLP sustavi uglavnom još uvijek teško postižu. Ovdje pokazujemo da značajno povećanje veličine jezičnih modela znatno poboljšava izvođenje zadataka s nekoliko primjera, ponekad čak i dosežući konkurentnost s prethodnim vrhunskim metodama fino podešavanja.  



Tl;dr


# Vježbe za nekoliko scenarija upotrebe  
1. Sažmi tekst  
2. Klasificiraj tekst  
3. Generiraj nova imena proizvoda


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasificiraj tekst  
#### Izazov  
Klasificirajte stavke u kategorije pružene u vrijeme izvođenja. U sljedećem primjeru pružamo i kategorije i tekst za klasifikaciju u promptu (*playground_reference). 

Upit korisnika: Bok, jedan od tipki na tipkovnici mog prijenosnog računala nedavno se pokvario i trebat će mi zamjena:

Klasificirana kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generirajte Nove Imena Proizvoda
#### Izazov
Izradite imena proizvoda iz primjera riječi. Ovdje u prompt uključujemo informacije o proizvodu za koji ćemo generirati imena. Također pružamo sličan primjer kako bismo pokazali obrazac koji želimo dobiti. Postavili smo i visoku vrijednost temperature kako bismo povećali slučajnost i dobili kreativnije odgovore.

Opis proizvoda: Kućni uređaj za pripremu milkshakea
Riječ-sjemenka: brz, zdrav, kompaktan.
Imena proizvoda: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis proizvoda: Par cipela koje mogu pristajati na bilo koju veličinu stopala.
Riječ-sjemenka: prilagodljiv, pristaje, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najbolje prakse za fino podešavanje GPT modela za klasifikaciju teksta](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za Više Pomoći  
[Tim za Komercijalizaciju OpenAI-ja](AzureOpenAITeam@microsoft.com) 


# Suradnici
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
